- https://www.neurosymbolic.org/papers/EllisWNSMHCST21.pdf
    - https://chatgpt.com/c/693b77a8-a330-832f-ae74-4a9ebfc828ab

In [1]:
"""
toy_dreamcoder.py

A minimal, practical DreamCoder-style system:
- Typed DSL (primitives + combinators)
- Probabilistic grammar (program prior)
- Wake: best-first enumerative program search
- Sleep: train a tiny task->grammar heuristic (logistic regression)
- Library learning: discover repeated subprograms and add them as new primitives

This is NOT the full DreamCoder implementation, but it reproduces the core loop:
  wake (solve) -> sleep (train recognizer) -> library learning (invent abstractions) -> repeat
"""

from __future__ import annotations
from dataclasses import dataclass
from typing import Any, Callable, Dict, List, Optional, Set, Tuple, Union
import heapq, math
import numpy as np

# -----------------------------
# Types (simple typed lambda calculus)
# -----------------------------
@dataclass(frozen=True)
class BaseType:
    name: str
    def __str__(self) -> str:
        return self.name

@dataclass(frozen=True)
class ArrowType:
    a: "Type"
    b: "Type"
    def __str__(self) -> str:
        return f"({self.a}->{self.b})"

Type = Union[BaseType, ArrowType]

def Arrow(a: Type, b: Type) -> ArrowType:
    return ArrowType(a, b)

Int = BaseType("int")
Bool = BaseType("bool")
ListInt = BaseType("list[int]")

Tii = Arrow(Int, Int)          # int -> int
Tib = Arrow(Int, Bool)         # int -> bool
Tll = Arrow(ListInt, ListInt)  # list[int] -> list[int]

# -----------------------------
# Expressions (AST)
# -----------------------------
@dataclass(frozen=True)
class Prim:
    name: str
    typ: Type

@dataclass(frozen=True)
class App:
    fn: "Expr"
    arg: "Expr"
    typ: Type

Expr = Union[Prim, App]

def expr_type(e: Expr) -> Type:
    return e.typ

def expr_str(e: Expr) -> str:
    if isinstance(e, Prim):
        return e.name
    return f"({expr_str(e.fn)} {expr_str(e.arg)})"

# -----------------------------
# Environment (primitive semantics)
# -----------------------------
class PrimitiveEnv:
    def __init__(self) -> None:
        self.values: Dict[str, Any] = {}
        self.types: Dict[str, Type] = {}

    def add(self, name: str, typ: Type, value: Any) -> None:
        self.values[name] = value
        self.types[name] = typ

    def clone(self) -> "PrimitiveEnv":
        new = PrimitiveEnv()
        new.values = dict(self.values)
        new.types = dict(self.types)
        return new

def build_base_env() -> PrimitiveEnv:
    env = PrimitiveEnv()
    # elementwise functions
    env.add("add1", Tii, lambda x: x + 1)
    env.add("sub1", Tii, lambda x: x - 1)
    env.add("mul2", Tii, lambda x: x * 2)
    env.add("is_even", Tib, lambda x: (x % 2) == 0)

    # higher-order list combinators (curried)
    env.add("map", Arrow(Tii, Tll), lambda f: (lambda xs: [f(x) for x in xs]))
    env.add("filter", Arrow(Tib, Tll), lambda p: (lambda xs: [x for x in xs if p(x)]))

    # composition (curried)
    env.add("compII", Arrow(Tii, Arrow(Tii, Tii)),
            lambda f: (lambda g: (lambda x: f(g(x)))))
    env.add("compLL", Arrow(Tll, Arrow(Tll, Tll)),
            lambda f: (lambda g: (lambda xs: f(g(xs)))))
    return env

def eval_expr(env: PrimitiveEnv, e: Expr) -> Any:
    if isinstance(e, Prim):
        return env.values[e.name]
    return eval_expr(env, e.fn)(eval_expr(env, e.arg))

# Semantics signatures (observational equivalence) for dedup during search
PROBE_INTS = [-2, -1, 0, 1, 2, 3]
PROBE_LISTS = [[], [0], [1, 2], [2, 3, 4], [-1, 0, 1]]

def signature(env: PrimitiveEnv, e: Expr) -> Tuple:
    t = expr_type(e)
    v = eval_expr(env, e)
    try:
        if t == Tii:
            return ("fii", tuple(v(x) for x in PROBE_INTS))
        if t == Tib:
            return ("fib", tuple(bool(v(x)) for x in PROBE_INTS))
        if t == Tll:
            return ("fll", tuple(tuple(v(xs)) for xs in PROBE_LISTS))
        return ("other", str(t), expr_str(e))
    except Exception:
        return ("err", str(t), expr_str(e))

# -----------------------------
# Grammar (program prior)
# -----------------------------
class Grammar:
    """
    A tiny probabilistic grammar:
    - Each primitive has a probability p(name).
    - Program "description length" is sum -log p(primitive) + app_penalty per application node.
    """
    def __init__(self, prim_names: List[str], app_penalty: float = 0.2) -> None:
        self.prob: Dict[str, float] = {p: 1.0 / len(prim_names) for p in prim_names}
        self.app_penalty = app_penalty

    def add_primitives(self, new_names: List[str], init_mass: float = 0.07) -> None:
        if not new_names:
            return
        scale = max(1e-9, 1.0 - init_mass * len(new_names))
        for k in list(self.prob.keys()):
            self.prob[k] *= scale
        for n in new_names:
            if n not in self.prob:
                self.prob[n] = init_mass
        Z = sum(self.prob.values())
        self.prob = {k: v / Z for k, v in self.prob.items()}

    def update_from_programs(self, programs: List[Expr], alpha: float = 0.5) -> None:
        # simple Dirichlet-smoothed MLE over primitive usage counts
        counts = {k: alpha for k in self.prob}

        def walk(e: Expr) -> None:
            if isinstance(e, Prim):
                if e.name in counts:
                    counts[e.name] += 1.0
            else:
                walk(e.fn)
                walk(e.arg)

        for p in programs:
            walk(p)

        Z = sum(counts.values())
        self.prob = {k: counts[k] / Z for k in counts}

    def prim_cost(self, name: str, task_logits: Optional[Dict[str, float]] = None) -> float:
        """
        If task_logits is provided, form a task-conditioned grammar:
          log p_task(prim) ∝ log p_base(prim) + task_logits(prim)
        """
        if task_logits is None:
            return -math.log(self.prob[name] + 1e-12)

        logits = {k: math.log(self.prob[k] + 1e-12) + task_logits.get(k, 0.0) for k in self.prob}
        m = max(logits.values())
        expv = {k: math.exp(logits[k] - m) for k in logits}
        Z = sum(expv.values())
        p = expv[name] / Z
        return -math.log(p + 1e-12)

# -----------------------------
# "Recognition model" (tiny + practical)
# -----------------------------
class Recognizer:
    """
    Toy version of DreamCoder recognition:
    from examples -> per-primitive logits (weakly) guiding the grammar.
    """
    def __init__(self, prim_names: List[str], feat_dim: int = 7) -> None:
        self.prim_names = prim_names
        self.W = np.zeros((len(prim_names), feat_dim), dtype=float)
        self.b = np.zeros((len(prim_names),), dtype=float)

    def featurize(self, examples: List[Tuple[List[int], List[int]]]) -> np.ndarray:
        lens_in = [len(i) for i, _ in examples]
        lens_out = [len(o) for _, o in examples]
        ratios = [lo / (li + 1e-6) for li, lo in zip(lens_in, lens_out)]

        deltas = []
        muls = []
        for inp, out in examples:
            if len(inp) == len(out) and len(inp) > 0:
                for a, b in zip(inp, out):
                    deltas.append(b - a)
                    muls.append(b / (a if a != 0 else 1.0))

        feat = np.array([
            float(np.mean(lens_in)),
            float(np.mean(lens_out)),
            float(np.mean(ratios)),
            float(np.mean(deltas)) if deltas else 0.0,
            float(np.mean(muls)) if muls else 0.0,
            1.0 if all(len(i) == len(o) for i, o in examples) else 0.0,   # map-ish
            1.0 if any(len(o) < len(i) for i, o in examples) else 0.0,    # filter-ish
        ], dtype=float)
        return feat

    def logits(self, examples: List[Tuple[List[int], List[int]]], strength: float = 0.1) -> Dict[str, float]:
        """
        Keep guidance weak; too-strong guidance can mislead enumerative search.
        """
        x = self.featurize(examples)
        z = self.W @ x + self.b
        return {name: float(strength * z[i]) for i, name in enumerate(self.prim_names)}

    def train(self, dataset: List[Tuple[List[Tuple[List[int], List[int]]], Expr]],
              lr: float = 0.1, steps: int = 300) -> None:
        for _ in range(steps):
            dW = np.zeros_like(self.W)
            db = np.zeros_like(self.b)
            for examples, prog in dataset:
                x = self.featurize(examples)
                used: Set[str] = set()

                def walk(e: Expr) -> None:
                    if isinstance(e, Prim):
                        used.add(e.name)
                    else:
                        walk(e.fn)
                        walk(e.arg)

                walk(prog)

                y = np.array([1.0 if n in used else 0.0 for n in self.prim_names], dtype=float)
                z = self.W @ x + self.b
                p = 1.0 / (1.0 + np.exp(-z))
                grad = (p - y)  # dL/dz

                dW += grad[:, None] * x[None, :]
                db += grad

            n = max(1, len(dataset))
            self.W -= lr * dW / n
            self.b -= lr * db / n

# -----------------------------
# Wake: enumerative search (best-first by description length)
# -----------------------------
class Synthesizer:
    def __init__(self, env: PrimitiveEnv, grammar: Grammar) -> None:
        self.env = env
        self.grammar = grammar

    def expr_cost(self, e: Expr, task_logits: Optional[Dict[str, float]]) -> float:
        if isinstance(e, Prim):
            return self.grammar.prim_cost(e.name, task_logits)
        return (self.expr_cost(e.fn, task_logits)
                + self.expr_cost(e.arg, task_logits)
                + self.grammar.app_penalty)

    def solve(self, target_type: Type, examples: List[Tuple[List[int], List[int]]],
              recognizer: Optional[Recognizer] = None, max_evals: int = 20000) -> Tuple[Optional[Expr], Dict[str, Any]]:
        task_logits = recognizer.logits(examples) if recognizer else None

        prims = [Prim(name, self.env.types[name]) for name in self.env.types]
        pq: List[Tuple[float, int, Expr]] = []
        uid = 0

        def push(e: Expr) -> None:
            nonlocal uid
            heapq.heappush(pq, (self.expr_cost(e, task_logits), uid, e))
            uid += 1

        for p in prims:
            push(p)

        bank_by_type: Dict[Type, List[Expr]] = {}
        funcs_by_arg: Dict[Type, List[Expr]] = {}
        seen_sem: Dict[Type, Set[Tuple]] = {}

        def add_to_bank(e: Expr) -> None:
            t = expr_type(e)
            bank_by_type.setdefault(t, []).append(e)
            if isinstance(t, ArrowType):
                funcs_by_arg.setdefault(t.a, []).append(e)

        def matches(e: Expr) -> bool:
            f = eval_expr(self.env, e)
            for inp, out in examples:
                try:
                    got = f(list(inp))
                except Exception:
                    return False
                if list(got) != list(out):
                    return False
            return True

        evaluated = 0
        while pq and evaluated < max_evals:
            _, _, e = heapq.heappop(pq)
            t = expr_type(e)
            sem = signature(self.env, e)
            if sem in seen_sem.setdefault(t, set()):
                continue

            seen_sem[t].add(sem)
            add_to_bank(e)
            evaluated += 1

            if t == target_type and matches(e):
                return e, {"evaluated": evaluated}

            # Expand
            if isinstance(t, ArrowType):
                for arg in bank_by_type.get(t.a, []):
                    push(App(e, arg, t.b))

            for fn in funcs_by_arg.get(t, []):
                ft = expr_type(fn)
                if isinstance(ft, ArrowType) and ft.a == t:
                    push(App(fn, e, ft.b))

        return None, {"evaluated": evaluated, "failed": True}

# -----------------------------
# Library learning: invent abstractions (macros)
# -----------------------------
def collect_subtrees(e: Expr) -> List[Expr]:
    out = [e]
    if isinstance(e, App):
        out += collect_subtrees(e.fn)
        out += collect_subtrees(e.arg)
    return out

def invent_macros(programs: List[Expr], min_count: int = 2) -> List[Expr]:
    """
    Practical variant: find repeated exact subtrees and add as new primitives.
    """
    counter: Dict[Tuple[str, str], int] = {}
    rep: Dict[Tuple[str, str], Expr] = {}

    for p in programs:
        for st in collect_subtrees(p):
            if isinstance(st, Prim):
                continue
            key = (str(expr_type(st)), expr_str(st))
            counter[key] = counter.get(key, 0) + 1
            rep[key] = st

    candidates = []
    for key, cnt in counter.items():
        t, s = key
        if cnt >= min_count and (t == str(Tll) or t == str(Tii)):
            candidates.append((cnt, -len(s), key))

    candidates.sort(reverse=True)
    chosen: List[Expr] = []
    for _, _, key in candidates:
        chosen.append(rep[key])
        if len(chosen) >= 2:
            break
    return chosen

# -----------------------------
# Demo tasks
# -----------------------------
def make_task(fn: Callable[[List[int]], List[int]], name: str) -> Tuple[str, List[Tuple[List[int], List[int]]]]:
    ins = [[1, 2, 3], [0, 5, -1], [2, 4]]
    return name, [(i, fn(i)) for i in ins]

TRAIN_TASKS = [
    make_task(lambda xs: [x + 1 for x in xs], "map_add1"),
    make_task(lambda xs: [x * 2 for x in xs], "map_mul2"),
    make_task(lambda xs: [x - 1 for x in xs], "map_sub1"),
    make_task(lambda xs: [(x + 1) * 2 for x in xs], "map_add1_then_mul2"),
    make_task(lambda xs: [(x + 1) * 2 for x in xs if (x % 2) == 0], "filter_even_then_add1_mul2"),
    make_task(lambda xs: [(x + 1) * 2 for x in xs], "map_add1_then_mul2_2"),
]

TEST_TASKS = [
    make_task(lambda xs: [(x + 1) * 2 for x in xs], "map_add1_then_mul2_test"),
    make_task(lambda xs: [(x + 1) * 2 for x in xs if x % 2 == 0], "filter_even_then_add1_mul2_test"),
]

def demo() -> None:
    base_env = build_base_env()
    grammar = Grammar(list(base_env.types.keys()))
    synth = Synthesizer(base_env, grammar)
    recognizer = Recognizer(list(base_env.types.keys()))

    print("== Phase 0: baseline on TEST (no learning) ==")
    for name, ex in TEST_TASKS:
        prog, info = synth.solve(Tll, ex, recognizer=None, max_evals=12000)
        print(f"{name:35s} evals={info['evaluated']:4d}  program={expr_str(prog) if prog else None}")

    print("\n== Phase 1: WAKE on TRAIN, collect solutions ==")
    solved: List[Tuple[List[Tuple[List[int], List[int]]], Expr]] = []
    for name, ex in TRAIN_TASKS:
        prog, info = synth.solve(Tll, ex, recognizer=None, max_evals=12000)
        assert prog is not None
        solved.append((ex, prog))
        print(f"{name:35s} evals={info['evaluated']:4d}  program={expr_str(prog)}")

    print("\n== Phase 2: SLEEP: train recognizer + update grammar ==")
    programs = [p for _, p in solved]
    grammar.update_from_programs(programs)
    recognizer.train(solved)
    for k, v in sorted(grammar.prob.items(), key=lambda kv: -kv[1])[:8]:
        print(f"  {k:10s} p={v:.3f}")

    print("\n== Phase 3: LIBRARY LEARNING: invent abstractions (macros) ==")
    macros = invent_macros(programs, min_count=2)
    learned_env = base_env.clone()
    new_names = []
    for i, m in enumerate(macros, start=1):
        name = f"abstr{i}"
        learned_env.add(name, expr_type(m), eval_expr(base_env, m))
        new_names.append(name)
        print(f"Invented {name}: type={expr_type(m)}  def={expr_str(m)}")

    grammar.add_primitives(new_names, init_mass=0.07)
    synth_learned = Synthesizer(learned_env, grammar)

    print("\n== Phase 4: evaluate on TEST again ==")
    print("-- (A) learned library only --")
    for name, ex in TEST_TASKS:
        prog, info = synth_learned.solve(Tll, ex, recognizer=None, max_evals=12000)
        print(f"{name:35s} evals={info['evaluated']:4d}  program={expr_str(prog) if prog else None}")

    print("-- (B) learned library + recognizer (weak guidance) --")
    for name, ex in TEST_TASKS:
        prog, info = synth_learned.solve(Tll, ex, recognizer=recognizer, max_evals=12000)
        print(f"{name:35s} evals={info['evaluated']:4d}  program={expr_str(prog) if prog else None}")

if __name__ == "__main__":
    demo()


== Phase 0: baseline on TEST (no learning) ==
map_add1_then_mul2_test             evals=  38  program=(map ((compII mul2) add1))
filter_even_then_add1_mul2_test     evals= 142  program=((compLL (map ((compII mul2) add1))) (filter is_even))

== Phase 1: WAKE on TRAIN, collect solutions ==
map_add1                            evals=   9  program=(map add1)
map_mul2                            evals=  11  program=(map mul2)
map_sub1                            evals=  10  program=(map sub1)
map_add1_then_mul2                  evals=  38  program=(map ((compII mul2) add1))
filter_even_then_add1_mul2          evals= 142  program=((compLL (map ((compII mul2) add1))) (filter is_even))
map_add1_then_mul2_2                evals=  38  program=(map ((compII mul2) add1))

== Phase 2: SLEEP: train recognizer + update grammar ==
  map        p=0.260
  add1       p=0.180
  mul2       p=0.180
  compII     p=0.140
  sub1       p=0.060
  is_even    p=0.060
  filter     p=0.060
  compLL     p=0.060

== Phas